# FinChain DPO RunPod Notebook

Goal: run the offline DPO comparator on FinChain as cheaply as possible. This notebook is deliberately explicit because DPO is the fixed-pair baseline, not the OPD/GRPO online loop.

Weighted DPO in this repo means per-pair scalar weighting before the batch reduction. The plain loss is `mean(loss_i)`; weighted DPO is `sum(w_i * loss_i) / sum(w_i)`. That is useful for OPD bridge pairs if you decide ambiguous prompts should dominate, but it is not what makes OPD online.

## RunPod Terminal Preflight
Run this in a JupyterLab terminal before opening the notebook cells:

```bash
cd /workspace
if [ ! -d finpost ]; then
  git clone https://github.com/shannan-liu1/finpost.git
fi
cd /workspace/finpost
git pull --ff-only
pip install -e ".[dev,rlvr]"
python -c "import finpost, torch; print('finpost ok'); print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')"

export FINPOST_FINCHAIN_TRAIN_JSONL=/workspace/data/finchain/train.jsonl
export FINPOST_FINCHAIN_VALIDATION_JSONL=/workspace/data/finchain/validation.jsonl
export FINPOST_FINCHAIN_TEST_JSONL=/workspace/data/finchain/test.jsonl
export WANDB_MODE=offline
```

If `import finpost` fails after `pip install -e`, use the fallback from `docs/runbooks/runpod-bootstrap.md`:

```bash
echo "/workspace/finpost/src" > /usr/local/lib/python3.11/dist-packages/finpost.pth
python -c "import finpost; print(finpost.__file__)"
```

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

REPO = Path('/workspace/finpost') if Path('/workspace/finpost').exists() else Path.cwd()
os.chdir(REPO)

MODEL_ID = 'Qwen/Qwen2.5-1.5B'
SFT_CONFIG = 'experiments/finchain/qwen25_1_5b_sft_runpod.yaml'
SFT_RUN_ROOT = Path('results/checkpoints/qwen25-1p5b-finchain-sft')
SFT_HF_DIR = Path('results/checkpoints/qwen25-1p5b-finchain-sft-hf')
DPO_CONFIG = 'experiments/dpo/finchain_qwen25_1_5b.yaml'
PAIR_RUN = Path('results/finchain_pairs/run_001')

os.environ.setdefault('WANDB_MODE', 'offline')


def run(cmd: str, *, check: bool = True) -> subprocess.CompletedProcess:
    print('$', cmd)
    return subprocess.run(cmd, shell=True, check=check, text=True)

print('repo:', REPO)
print('WANDB_MODE:', os.environ.get('WANDB_MODE'))

In [ ]:
import torch

print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu count:', torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(idx, props.name, round(props.total_memory / 1024**3, 1), 'GB')
else:
    print('No CUDA GPU visible. Stop here on RunPod.')

In [ ]:
from finpost.data.finchain import load_finchain, resolve_finchain_path

for split in ['train', 'validation', 'test']:
    path = resolve_finchain_path(split)
    print(split, '->', path, 'exists=', path.exists())

train_examples = load_finchain('train')
print('train examples:', len(train_examples))
print('first prompt id:', train_examples[0].id)
print('first gold answer:', train_examples[0].final_answer)

## Step 1 - Optional SFT Baseline
DPO assumes a policy checkpoint and a frozen reference checkpoint. If you already have an HF-format SFT checkpoint, set `SFT_HF_DIR` above and skip this.

Single GPU terminal command:

```bash
cd /workspace/finpost
WANDB_MODE=offline python -m finpost.training.train \
  --config experiments/finchain/qwen25_1_5b_sft_runpod.yaml \
  --device cuda

python scripts/convert_checkpoint_to_hf.py \
  --checkpoint-dir results/checkpoints/qwen25-1p5b-finchain-sft/step-00001000 \
  --base-model-id Qwen/Qwen2.5-1.5B \
  --out-dir results/checkpoints/qwen25-1p5b-finchain-sft-hf \
  --dtype bfloat16
```

Cost control: first run `--max-steps 20` on the train command. Only run the full 1000-step config after the loss and checkpoint path look sane.

In [ ]:
print('SFT canary command:')
print('WANDB_MODE=offline python -m finpost.training.train --config', SFT_CONFIG, '--device cuda --max-steps 20')
print('
Full SFT command:')
print('WANDB_MODE=offline python -m finpost.training.train --config', SFT_CONFIG, '--device cuda')
print('
Convert command after choosing a step directory:')
print('python scripts/convert_checkpoint_to_hf.py --checkpoint-dir', SFT_RUN_ROOT / 'step-00001000', '--base-model-id', MODEL_ID, '--out-dir', SFT_HF_DIR, '--dtype bfloat16')

## Step 2 - Build FinChain Preference Pairs
This is offline by design: sample completions once, grade them, write `pairs.jsonl`, then train. Use small values first; pair generation is the expensive part.

In [ ]:
single_gpu_pair_cmd = f"""
CUDA_VISIBLE_DEVICES=0 python scripts/build_dpo_pairs.py \
  --model-checkpoint {SFT_HF_DIR} \
  --sources finchain \
  --out-dir {PAIR_RUN / 'single_gpu'} \
  --heldout-train-n 256 \
  --samples-per-prompt 4 \
  --generation-batch-size 16 \
  --max-new-tokens-finchain 512 \
  --max-pairs-per-prompt 4
""".strip()
print(single_gpu_pair_cmd)

## Two-GPU Rollout Parallelism
This uses both A40s without distributed training complexity. Each process owns one GPU and a deterministic prompt shard. Merge before training.

Terminal command:

```bash
cd /workspace/finpost
export CKPT=results/checkpoints/qwen25-1p5b-finchain-sft-hf
export OUT=results/finchain_pairs/run_001

CUDA_VISIBLE_DEVICES=0 python scripts/build_dpo_pairs.py --model-checkpoint $CKPT --sources finchain --out-dir $OUT/shards/shard-00-of-02 --heldout-train-n 2000 --samples-per-prompt 8 --generation-batch-size 64 --max-new-tokens-finchain 768 --max-pairs-per-prompt 8 --shard-id 0 --num-shards 2 &
CUDA_VISIBLE_DEVICES=1 python scripts/build_dpo_pairs.py --model-checkpoint $CKPT --sources finchain --out-dir $OUT/shards/shard-01-of-02 --heldout-train-n 2000 --samples-per-prompt 8 --generation-batch-size 64 --max-new-tokens-finchain 768 --max-pairs-per-prompt 8 --shard-id 1 --num-shards 2 &
wait

python scripts/merge_dpo_pair_shards.py --shard-dirs $OUT/shards/shard-00-of-02 $OUT/shards/shard-01-of-02 --out-dir $OUT/merged
```

In [ ]:
two_gpu_pair_cmd = f"""
export CKPT={SFT_HF_DIR}
export OUT={PAIR_RUN}
CUDA_VISIBLE_DEVICES=0 python scripts/build_dpo_pairs.py --model-checkpoint $CKPT --sources finchain --out-dir $OUT/shards/shard-00-of-02 --heldout-train-n 2000 --samples-per-prompt 8 --generation-batch-size 64 --max-new-tokens-finchain 768 --max-pairs-per-prompt 8 --shard-id 0 --num-shards 2 &
CUDA_VISIBLE_DEVICES=1 python scripts/build_dpo_pairs.py --model-checkpoint $CKPT --sources finchain --out-dir $OUT/shards/shard-01-of-02 --heldout-train-n 2000 --samples-per-prompt 8 --generation-batch-size 64 --max-new-tokens-finchain 768 --max-pairs-per-prompt 8 --shard-id 1 --num-shards 2 &
wait
python scripts/merge_dpo_pair_shards.py --shard-dirs $OUT/shards/shard-00-of-02 $OUT/shards/shard-01-of-02 --out-dir $OUT/merged
""".strip()
print(two_gpu_pair_cmd)

## Step 3 - Train DPO
Repo-native DPO is pedagogical and single-process. It is efficient for the 1.5B path because reference log-probs are precomputed once. Do not launch this with `accelerate`; use TRL or Axolotl if you want true distributed DPO training.

In [ ]:
dpo_canary = f"WANDB_MODE=offline python -m finpost.training.dpo_train --config {DPO_CONFIG} --device cuda --max-steps 20"
dpo_full = f"WANDB_MODE=offline python -m finpost.training.dpo_train --config {DPO_CONFIG} --device cuda"
print('DPO canary:')
print(dpo_canary)
print('
DPO full:')
print(dpo_full)

## Step 4 - Evaluate
Convert the DPO checkpoint to HF format, then run the FinChain eval wrapper.

```bash
python scripts/convert_checkpoint_to_hf.py \
  --checkpoint-dir results/checkpoints/qwen25-1p5b-finchain-dpo/step-00000500 \
  --base-model-id Qwen/Qwen2.5-1.5B \
  --out-dir results/checkpoints/qwen25-1p5b-finchain-dpo-hf \
  --dtype bfloat16

python scripts/run_finchain_eval.py \
  --checkpoints sft=results/checkpoints/qwen25-1p5b-finchain-sft-hf dpo=results/checkpoints/qwen25-1p5b-finchain-dpo-hf \
  --n 500 \
  --out-dir results/evals/finchain_dpo_run_001 \
  --device cuda
```

Exit gate: DPO is worth keeping only if it improves held-out FinChain accuracy or gives a useful negative result against OPD/GRPO for the same cost.